# Specific Task 1 — Quark/Gluon Classification with Contrastive Learning

This notebook implements a two-phase pipeline for quark/gluon jet classification
using contrastive self-supervised learning on graph representations.

**Phase 1 — SimCLR pretraining (unsupervised):**
Two augmented views of each jet graph are generated and a GNN encoder is trained
to produce similar embeddings for views of the same jet, and dissimilar embeddings
for different jets. No labels are used in this phase.

**Phase 2 — Linear probe (supervised):**
The pretrained encoder is frozen and a single linear layer is trained on top
to classify quarks and gluons using the learned representations.

**Why contrastive learning?**
Contrastive learning allows the model to learn structure from unlabeled data —
directly relevant to the GSoC anomaly detection project, where labeled anomalous
events are unavailable and the model must learn what "normal" looks like
from background data alone.

The graph construction reuses the same pipeline from Common Task 2:
kNN edges in (η, φ) space with 6 node features and 4 edge features per node.

In [3]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader as TorchDataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import EdgeConv, SAGEConv, global_mean_pool, global_max_pool

In [4]:
graphs = torch.load("datasets/qg_graph_dataset_k8_edgeattr.pt", weights_only=False)
y = []
for g in graphs:
    y.append(g.y.item())
y = np.array(y)

In [37]:
TEMPERATURE = 0.5
PROJ_DIM = 128
HIDDEN_DIM = 64

# set division
VAL_SIZE = 0.15
TEST_SIZE = 0.15

EPOCHS = 20
BASELINE_EPOCHS = 8
BATCH_SIZE = 256

LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [41]:
#stratified split
train_idx, temp_idx = train_test_split(np.arange(len(graphs)), test_size=VAL_SIZE + TEST_SIZE, stratify=y)

val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y[temp_idx])

In [42]:
subset_idx = train_test_split(
    np.arange(len(graphs)),
    train_size=20000,
    stratify=y,
    random_state=42
)[0]

subset_graphs = [graphs[i] for i in subset_idx]
subset_y = y[subset_idx]

In [43]:
train_graphs = [graphs[i] for i in train_idx]
val_graphs = [graphs[i] for i in val_idx]
test_graphs = [graphs[i] for i in test_idx]

In [44]:
val_loader  = PyGDataLoader(val_graphs,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = PyGDataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Graph Augmentations

SimCLR requires two different augmented views of the same input.
For jet graphs, we use two physically motivated augmentations:

- **Node feature dropout:** zeroes out all features of randomly selected nodes
  with probability `p_node`. Simulates detector noise or missing deposits.
- **Edge dropout:** randomly removes edges with probability `p_edge`.
  Simulates variation in the local connectivity structure of the jet.

Each call to `__getitem__` applies both augmentations independently twice,
producing two different views `(view1, view2)` of the same jet graph.

In [45]:
class SimCLRGraphDataset(Dataset):
    def __init__(self, graphs, p_node=0.1, p_edge=0.1):
        self.graphs = graphs
        self.p_node = p_node
        self.p_edge = p_edge
    
    def __len__(self):
        return len(self.graphs)
    
    def __getitem__(self, idx):
        data =self.graphs[idx]
        view1 = self.augment(data)
        view2 = self.augment(data)
        return view1, view2
    
    def augment(self, data):
        data = self.augment_node_features(data)
        data = self.augment_edges(data)
        return data

    def augment_node_features(self, data):
        data_copy = data.clone()
        mask = torch.bernoulli(torch.full((data_copy.num_nodes, 1), 1 - self.p_node))
        data_copy.x = data_copy.x * mask
        return data_copy

    def augment_edges(self, data):
        data_copy = data.clone()
        mask = torch.bernoulli(torch.full((data_copy.num_edges, ), 1 - self.p_edge))
        data_copy.edge_index = data_copy.edge_index[:, mask.bool()]
        data_copy.edge_attr = data_copy.edge_attr[mask.bool()]
        return data_copy

In [46]:
sub_train_idx, sub_temp_idx = train_test_split(
    np.arange(len(subset_graphs)),
    test_size=VAL_SIZE + TEST_SIZE,
    stratify=subset_y,
    random_state=42
)
sub_val_idx, sub_test_idx = train_test_split(
    sub_temp_idx,
    test_size=0.5,
    stratify=subset_y[sub_temp_idx],
    random_state=42
)

sub_train_graphs = [subset_graphs[i] for i in sub_train_idx]
sub_val_graphs   = [subset_graphs[i] for i in sub_val_idx]
sub_test_graphs  = [subset_graphs[i] for i in sub_test_idx]

train_dataset = SimCLRGraphDataset(sub_train_graphs)
val_dataset   = SimCLRGraphDataset(sub_val_graphs)

In [47]:
from torch_geometric.data import Batch

def collate_fn(batch):
    batch_view1, batch_view2 = zip(*batch)
    batch_view1 = list(batch_view1)
    batch_view2 = list(batch_view2)
    batch_view1 = Batch.from_data_list(batch_view1)
    batch_view2 = Batch.from_data_list(batch_view2)
    return batch_view1, batch_view2

In [48]:
train_loader = PyGDataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [49]:
batch_view1, batch_view2 = next(iter(train_loader))
print(batch_view1)
print(batch_view2)

DataBatch(x=[179079, 6], edge_index=[2, 1376477], edge_attr=[1376477, 4], y=[256], pos=[179079, 2], batch=[179079], ptr=[257])
DataBatch(x=[179079, 6], edge_index=[2, 1375984], edge_attr=[1375984, 4], y=[256], pos=[179079, 2], batch=[179079], ptr=[257])


## Model Architecture

The SimCLR model consists of two components:

**GNNEncoder:** two EdgeConv layers with BatchNorm and dual global pooling
(mean + max concatenated), producing a fixed-size embedding per graph.
This is the same architecture used in Common Task 2.

**ProjectionHead:** a two-layer MLP with BatchNorm that maps the encoder
embedding to a lower-dimensional space where the contrastive loss is computed.
Following the original SimCLR paper, the projection head is discarded after
pretraining — only the encoder is used for downstream classification.

In [50]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels = 64):
        super().__init__()

        self.conv1 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * in_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU()
            )
        )

        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU()
            )
        )

        self.bn2 = nn.BatchNorm1d(hidden_channels)
    
    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)

        x_mean_pool = global_mean_pool(x, batch)
        x_max_pool = global_max_pool(x, batch)

        x = torch.cat([x_mean_pool, x_max_pool], dim=1)

        return x
    

In [51]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x):
        x = self.mlp(x)
        return x

In [52]:
class SimCLRModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, proj_dim):
        super().__init__()
        self.encoder = GNNEncoder(in_channels, hidden_channels)
        self.projector = ProjectionHead(2 * hidden_channels, proj_dim)

    def forward(self, x, edge_index, batch):
        x = self.encoder(x, edge_index, batch)
        x = self.projector(x)
        x_norm = F.normalize(x, dim=1)
        return x_norm

In [53]:
in_channels = graphs[0].x.shape[1]
CLR_model = SimCLRModel(in_channels=in_channels, hidden_channels=HIDDEN_DIM, proj_dim=PROJ_DIM).to(DEVICE)

## NT-Xent Loss

The NT-Xent (Normalized Temperature-scaled Cross Entropy) loss is the core
of SimCLR. Given a batch of N jet graphs, each producing two views,
the loss encourages the model to:

- Pull together embeddings of the two views of the same jet (positive pairs)
- Push apart embeddings of views from different jets (negative pairs)

The temperature parameter `τ` controls the sharpness of the distribution —
lower values make the model more selective in distinguishing pairs.

With batch size B, each sample has 1 positive pair and 2B-2 negative pairs.
Larger batches provide more negatives and generally lead to better representations,
which is why SimCLR benefits significantly from large batch sizes and GPU training.

In [54]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        sim_matrix = torch.cat([z1, z2], dim=0)
        sim_matrix = (sim_matrix @ sim_matrix.T / self.temperature)
        mask = torch.eye(sim_matrix.shape[0]).to(z1.device)
        mask = mask.to(torch.bool)
        sim_matrix = sim_matrix.masked_fill(mask, float('-inf'))

        batch_size = z1.shape[0]
        pos_labels = torch.cat([torch.arange(batch_size, 2 * batch_size),torch.arange(0, batch_size)], dim=0)

        loss = F.cross_entropy(sim_matrix, pos_labels.to(z1.device))
        return loss

In [55]:
CLR_optimizer = torch.optim.Adam(CLR_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
CLR_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(CLR_optimizer, mode='max')
CLR_criterion = NTXentLoss(temperature=TEMPERATURE)


## Phase 1 — SimCLR Pretraining

The encoder is trained unsupervised on a stratified subset of 20,000 events.
The full dataset of 139,306 events was reduced due to CPU memory and time constraints.

Training is monitored by the NT-Xent validation loss. No accuracy or AUC is computed
in this phase — the goal is to learn useful representations, not to classify.

**Hardware note:** Full SimCLR convergence (NT-Xent loss < 2.5) requires GPU.
For the GSoC project, training will be conducted on Google Colab (T4/A100),
which is expected to reduce training time by 10-20x and allow proper convergence
within the proposed timeline.

In [56]:
def train_one_epoch_CLR(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_graphs = 0
    
    for batch in tqdm(loader, desc=f"Training", leave=False):
        view1, view2 = batch

        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)

        optimizer.zero_grad()
        z1 = model(view1.x, view1.edge_index, view1.batch)
        z2 = model(view2.x, view2.edge_index, view2.batch)
        loss = criterion(z1,z2)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item() * view1.num_graphs
        total_graphs += view1.num_graphs

    return total_loss / total_graphs

In [57]:
@torch.no_grad()
def evaluate_CLR(model, loader, criterion):
    model.eval()

    total_loss = 0
    total_graphs = 0

    for view1, view2 in loader:
        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)

        z1 = model(view1.x, view1.edge_index, view1.batch)
        z2 = model(view2.x, view2.edge_index, view2.batch)
        loss = criterion(z1,z2)

        total_loss += loss.item() * view1.num_graphs
        total_graphs += view1.num_graphs
    return total_loss / total_graphs

In [58]:
val_dataset = SimCLRGraphDataset(val_graphs)
val_loader_CLR = PyGDataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [59]:
best_val_loss = float("inf")
patience = 3
patience_counter = 0
best_model_path = "models/best_CLR.pt"

In [61]:
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_CLR(CLR_model, train_loader, CLR_optimizer, CLR_criterion)
    val_loss = evaluate_CLR(CLR_model, val_loader_CLR, CLR_criterion)
    CLR_scheduler.step(val_loss)

    history.append({
        'epoch': epoch,
        'val_loss': val_loss,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(CLR_model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

Epoch 01/20 | train_loss=4.9270 | val_loss=4.8210


Epoch 02/20 | train_loss=4.7416 | val_loss=4.7085


Epoch 03/20 | train_loss=4.6659 | val_loss=4.6550


Epoch 04/20 | train_loss=4.6199 | val_loss=4.6216


Epoch 05/20 | train_loss=4.5831 | val_loss=4.5803


...

Epoch 16/20 | train_loss=4.4056 | val_loss=4.4173


Epoch 17/20 | train_loss=4.4053 | val_loss=4.4147


Epoch 18/20 | train_loss=4.4042 | val_loss=4.4158


Epoch 19/20 | train_loss=4.4031 | val_loss=4.4128


Epoch 20/20 | train_loss=4.4027 | val_loss=4.4117


In [62]:
CLR_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

# congela os pesos do encoder
for param in CLR_model.encoder.parameters():
    param.requires_grad = False

## Phase 2 — Linear Probe

The pretrained encoder is frozen — its weights are not updated during this phase.
A single linear layer is trained on top to classify quarks and gluons.

This is the standard evaluation protocol from the SimCLR paper: if the encoder
learned useful representations, a linear classifier should perform well without
needing to fine-tune the entire network. The quality of the representations
is directly reflected in the linear probe accuracy.

In [64]:
class LinClassifier(nn.Module):
    def __init__(self, encod, num_classes = 2):
        super().__init__()

        self.encod = encod
        self.classifier = nn.Linear(2 * HIDDEN_DIM, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.encod(x, edge_index, batch)
        x = self.classifier(x)
        return x
        

In [ ]:
classifier = LinClassifier(CLR_model.encoder).to(DEVICE)

In [68]:
class_optimizer = torch.optim.Adam(classifier.classifier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
class_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(class_optimizer, mode='max')
class_criterion = nn.CrossEntropyLoss()

In [73]:
def train_one_epoch_gnn(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_graphs = 0
    
    for batch in tqdm(loader, desc=f"Training", leave=False):
        batch = batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(logits, batch.y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        total_graphs += batch.num_graphs

    return total_loss / total_graphs

In [74]:
@torch.no_grad()
def evaluate_gnn(model, loader):
    model.eval()

    all_y_true = []
    all_y_pred = []
    all_y_prob = []

    for batch in loader:
        batch = batch.to(DEVICE)

        logits = model(batch.x, batch.edge_index, batch.batch)
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        all_y_true.append(batch.y.cpu().numpy())
        all_y_pred.append(preds.cpu().numpy())
        all_y_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(all_y_true)
    y_pred = np.concatenate(all_y_pred)
    y_prob = np.concatenate(all_y_prob)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    pos_rate = y_pred.mean()

    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc,
        "pos_rate": pos_rate,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

In [75]:
best_val_auc = -1
patience = 3
patience_counter = 0
best_model_path = "models/best_LinClassifier.pt"

In [76]:
train_loader_linear = PyGDataLoader(sub_train_graphs, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

In [77]:
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_gnn(classifier, train_loader_linear, class_optimizer,class_criterion)
    val_metrics = evaluate_gnn(classifier, val_loader)
    class_scheduler.step(val_metrics["auc"])

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_auc': val_metrics['auc'],
        'val_f1': val_metrics['f1'],
        'val_acc': val_metrics['accuracy']
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_auc={val_metrics['auc']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
    )

    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        patience_counter = 0
        torch.save(classifier.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

Epoch 01/20 | train_loss=0.6946 | val_auc=0.5137 | val_f1=0.3075 | val_acc=0.5055 | 


Epoch 02/20 | train_loss=0.6926 | val_auc=0.5447 | val_f1=0.3085 | val_acc=0.5187 | 


Epoch 03/20 | train_loss=0.6907 | val_auc=0.5733 | val_f1=0.3444 | val_acc=0.5377 | 


Epoch 04/20 | train_loss=0.6889 | val_auc=0.5906 | val_f1=0.4076 | val_acc=0.5578 | 


Epoch 05/20 | train_loss=0.6873 | val_auc=0.6004 | val_f1=0.4451 | val_acc=0.5694 | 


...

Epoch 16/20 | train_loss=0.6740 | val_auc=0.6311 | val_f1=0.5536 | val_acc=0.5933 | 


Epoch 17/20 | train_loss=0.6731 | val_auc=0.6311 | val_f1=0.5517 | val_acc=0.5933 | 


Epoch 18/20 | train_loss=0.6722 | val_auc=0.6322 | val_f1=0.5505 | val_acc=0.5939 | 


Epoch 19/20 | train_loss=0.6713 | val_auc=0.6335 | val_f1=0.5569 | val_acc=0.5945 | 


Epoch 20/20 | train_loss=0.6705 | val_auc=0.6349 | val_f1=0.5608 | val_acc=0.5957 | 


In [78]:
classifier.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_metrics = evaluate_gnn(classifier, test_loader)

print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"F1: {test_metrics['f1']:.4f}")
print(f"AUC: {test_metrics['auc']:.4f}")

Accuracy: 0.5972
F1: 0.5608
AUC: 0.6324


## Results and Discussion

### Performance comparison

| Model | AUC | Accuracy | F1 |
|-------|-----|----------|----|
| EdgeConv GNN — supervised (Common Task 2) | 0.779 | 0.712 | 0.717 |
| SimCLR + linear probe (Specific Task 1) | 0.632 | 0.597 | 0.561 |

### Discussion

**The gap between supervised and contrastive learning is expected.**
The SimCLR pretraining converged to NT-Xent loss ~4.40, far from the ideal ~2.0-2.5
achievable with GPU and larger batch sizes. The representations learned under these
constraints are less discriminative than a fully supervised model trained on the
complete dataset.

**Despite the gap, the pipeline is correct and complete.**
The model learned representations that perform significantly above chance (AUC 0.632 vs 0.5),
demonstrating that the contrastive objective extracted useful structure from the jet graphs
without any labels.

**Limitations:**
- Pretraining used only 20,000 events due to CPU constraints
- NT-Xent loss did not converge to optimal values — GPU required
- Linear probe trained on the same 20k subset, vs 97k for the supervised model
- Larger batch sizes (512-1024) would provide more negatives per step and improve convergence

**Connection to the GSoC project:**
This pipeline is a direct prototype of the contrastive anomaly detection framework
proposed for ML4SCI GENIE. In the anomaly detection setting, the encoder would be
trained on QCD background jets only, and anomaly scores would be computed as the
distance of test embeddings from the learned normal distribution — following
the approach of Luo et al. (2022). The supervised classification task here
validates the pipeline before extending it to the unsupervised anomaly detection setting.

Full convergence on the complete LHC Olympics 2020 dataset with GPU resources
is planned as part of the GSoC timeline.